In [ ]:
import numpy as np
import tensorflow as tf
print(tf.__version__) 
print(tf.keras.__version__) 

In [ ]:
from tensorflow.keras import mixed_precision
import tensorflow as tf

# Check if mixed precision can be set
policy = mixed_precision.Policy('mixed_float16')
mixed_precision.set_global_policy(policy)

print("Mixed precision set to:", mixed_precision.global_policy())

In [ ]:
from tensorflow.keras import mixed_precision
import os
from tflite_model_maker import object_detector
from tflite_model_maker import model_spec
from sklearn.model_selection import train_test_split
import shutil
import glob

Pisah annotation & image

In [ ]:
import os
import glob
import shutil
from tensorflow.keras import mixed_precision

# Set mixed precision policy (correct for TensorFlow >= 2.9)
policy = mixed_precision.Policy('mixed_float16')
mixed_precision.set_global_policy(policy)

# Define paths to image and annotation directories
base_dir = "split/10-25m"
image_dir = os.path.join(base_dir, "train")  # Directory containing images and annotations
annotations_dir = os.path.join(base_dir, "annotations")  # Directory for XML annotations
images_output_dir = os.path.join(base_dir, "train/images")
annotations_output_dir = os.path.join(base_dir, "train/annotations")

# Ensure the output directories exist
os.makedirs(images_output_dir, exist_ok=True)
os.makedirs(annotations_output_dir, exist_ok=True)

# Get list of all image and annotation files
image_files = glob.glob(os.path.join(image_dir, "*.jpg"))  # Adjust for PNG if needed
annotation_files = glob.glob(os.path.join(image_dir, "*.xml"))  # Adjust for other formats if needed

# Move image files to the new images directory
for image_file in image_files:
    shutil.move(image_file, os.path.join(images_output_dir, os.path.basename(image_file)))

# Move annotation files to the new annotations directory
for annotation_file in annotation_files:
    shutil.move(annotation_file, os.path.join(annotations_output_dir, os.path.basename(annotation_file)))

print("Separation of images and annotations completed.")


Pisah data test, validation, train

In [ ]:
# Set mixed precision policy (correct for TensorFlow >= 2.9)
policy = mixed_precision.Policy('mixed_float16')
mixed_precision.set_global_policy(policy)

# Define paths to image and annotation directories
image_dir = os.path.join("DataSet (All)", "JPG")
annotations_dir = os.path.join("DataSet (All)", "XML")

# Create temporary directories for train, validation, and test splits
split_dirs = {
    'train': {'images': 'split/train/images', 'annotations': 'split/train/annotations'},
    'val': {'images': 'split/val/images', 'annotations': 'split/val/annotations'},
    'test': {'images': 'split/test/images', 'annotations': 'split/test/annotations'}
}

# Ensure the split directories exist
for split_type in split_dirs:
    os.makedirs(split_dirs[split_type]['images'], exist_ok=True)
    os.makedirs(split_dirs[split_type]['annotations'], exist_ok=True)

# Get list of all image and annotation files
image_files = glob.glob(os.path.join(image_dir, "*.jpg"))  # Adjust for PNG if needed
annotation_files = [os.path.join(annotations_dir, os.path.basename(f).replace(".jpg", ".xml")) for f in image_files]

# Make sure the number of images and annotations match
assert len(image_files) == len(annotation_files), "Mismatch between image and annotation files!"

# Split data into 70% train, 30% rest
train_image_files, rest_image_files, train_annotation_files, rest_annotation_files = train_test_split(
    image_files, annotation_files, test_size=0.3, random_state=42
)

# Split the rest into 20% validation and 10% test (2/3 validation, 1/3 test)
val_image_files, test_image_files, val_annotation_files, test_annotation_files = train_test_split(
    rest_image_files, rest_annotation_files, test_size=1/3, random_state=42
)

# Helper function to copy files to the correct split directories
def copy_files(image_files, annotation_files, split_type):
    for img, ann in zip(image_files, annotation_files):
        shutil.copy(img, split_dirs[split_type]['images'])
        shutil.copy(ann, split_dirs[split_type]['annotations'])

# Copy the split datasets into the corresponding directories
copy_files(train_image_files, train_annotation_files, 'train')
copy_files(val_image_files, val_annotation_files, 'val')
copy_files(test_image_files, test_annotation_files, 'test')
